# Google Colabでのファインチューニング(Elyza-7B)

## ステップ1: Google Colabにアクセス

1-1. ブラウザで https://colab.research.google.com/ を開く
Googleアカウントでログイン

1-2. 新しいノートブックを作成

画面左上の 「ファイル」 → 「ノートブックを新規作成」 をクリック
ノートブックの名前を変更:

画面上部の「Untitled0.ipynb」をクリック
「任意のプロジェクト名」に変更
Enterキーを押す

1-3. GPU設定

画面上部メニューの 「ランタイム」 → 「ランタイムのタイプを変更」 をクリック
ダイアログが開くので:

「ハードウェアアクセラレータ」を 「GPU」 に変更（「T4 GPU」を選択）

「保存」 をクリック

## ステップ2: 環境セットアップ
2-1. セルを作成
画面中央の 「+ コード」 ボタンをクリックして、新しいコードセルを作成
2-2. 以下のコードをコピー&ペースト

In [ ]:
"""
Elyza-7B v3 環境セットアップ
"""

print("環境セットアップ開始...")

import subprocess
import sys

# ステップ1: クリーンアップ
print("\n[1/3] クリーンアップ中...")
packages_to_remove = ['diffusers', 'transformers', 'accelerate', 'peft', 'trl', 'bitsandbytes']
for pkg in packages_to_remove:
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', pkg],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# ステップ2: パッケージインストール
print("\n[2/3] パッケージインストール中...")
packages = [
    'accelerate==0.27.2',
    'transformers==4.38.2',
    'peft==0.9.0',
    'trl==0.8.1',
    'bitsandbytes',
    'datasets==2.18.0',
    'sentencepiece==0.2.0',
    'protobuf==4.25.3',
    'PyYAML==6.0.1'
]

for pkg in packages:
    print(f"  インストール中: {pkg}")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', pkg], check=True)

# ステップ3: 確認
print("\n[3/3] インストール確認...")
import torch, transformers, peft, trl, bitsandbytes
print(f"✓ PyTorch: {torch.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ bitsandbytes: {bitsandbytes.__version__}")

if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ メモリ: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("GPUが検出されません！ランタイム設定を確認してください")

print("\n" + "=" * 70)
print("セットアップ完了!")
print("=" * 70)
print("\n⚠️  重要: 次の手順を必ず実行してください:")
print("   1. 画面上部メニュー「ランタイム」をクリック")
print("   2. 「セッションを再起動」を選択")
print("=" * 70)

"""
### 2-3. 実行
1. セル左側の **▶（再生ボタン）** をクリック
2. または、セルを選択して **Shift + Enter** を押す

### 2-4. 実行結果を確認
以下のような表示が出る
    
"""
"""    
✓ PyTorch: 2.9.0+cu126
✓ Transformers: 4.38.2
✓ PEFT: 0.9.0
✓ TRL: 0.8.1
✓ bitsandbytes: 0.49.0
✓ GPU: Tesla T4
✓ メモリ: 15.0 GB
"""

: 

## ステップ3: ランタイムの再起動
3-1. 再起動の実行

画面上部メニューの 「ランタイム」 をクリック
「セッションを再起動」 をクリック
確認ダイアログが出たら 「はい」 をクリック

3-2. 再起動完了の確認

画面左下に「接続済み」と表示される
セルの出力がクリアされる

この再起動を忘れると、後でエラーが出る

## ステップ4: Google Driveのマウント
4-1. 新しいセルを作成
画面左側の 「+ コード」 をクリック
4-2. 以下のコードをコピー&ペースト

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
work_dir = "/content/drive/MyDrive/Tech-Comply-AI-v2"
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print("=" * 70)
print("✓ Google Drive マウント完了")
print(f"✓ 作業ディレクトリ: {work_dir}")
print("=" * 70)

"""
### 4-3. 実行
▶ボタンをクリックまたは Shift + Enter

### 4-4. Google Driveの認証
1. **「Googleドライブに接続」** というリンクが表示される
2. クリックして、Googleアカウントを選択
3. **「許可」** をクリック

### 4-5. 完了確認

✓ Google Drive マウント完了
✓ 作業ディレクトリ: /content/drive/MyDrive/Tech-Comply-AI-v2
"""

## ステップ5: ファイルのアップロード
5-1. 新しいセルを作成
5-2. 以下のコードをコピー&ペースト

In [ ]:
from google.colab import files

print("=" * 70)
print("ファイルアップロード")
print("=" * 70)
print("\n以下の4つのファイルをアップロードしてください:")
print("1. train_data_natural_language.jsonl")
print("2. config_v3.yaml")
print("3. train_finetuning_v3.py")
print("4. inference_v3.py")
print("\n「ファイルを選択」ボタンが表示されるのを待ってください...")
print("=" * 70)
print()

uploaded = files.upload()

print("\n" + "=" * 70)
print("アップロード完了!")
print("=" * 70)
for filename in uploaded.keys():
    size_kb = len(uploaded[filename]) / 1024
    print(f"✓ {filename} ({size_kb:.1f} KB)")
print("=" * 70)

"""
### 5-3. 実行
▶ボタンをクリック

### 5-4. ファイルを選択
1. **「ファイルを選択」** ボタンが表示される
2. クリックして、以下の4ファイルを選択:
   - `train_data_natural_language.jsonl`
   - `config_v3.yaml`
   - `train_finetuning_v3.py`
   - `inference_v3.py`
3. **「開く」** をクリック

### 5-5. アップロード完了を待つ
アップロードバーが表示されます。完了すると:
```
✓ train_data_natural_language.jsonl (174 KB)
✓ config_v3.yaml (2 KB)
✓ train_finetuning_v3.py (8 KB)
✓ inference_v3.py (4 KB)
"""

## ステップ6: データの確認
6-1. 新しいセルを作成
6-2. 以下のコードをコピー&ペースト

In [ ]:
import os
import json

print("=" * 70)
print("ファイル確認")
print("=" * 70)

# 必須ファイルの確認
required_files = [
    'train_data_natural_language.jsonl',
    'config_v3.yaml', 
    'train_finetuning_v3.py',
    'inference_v3.py'
]

all_ok = True
for filename in required_files:
    if os.path.exists(filename):
        size_kb = os.path.getsize(filename) / 1024
        print(f"✓ {filename} ({size_kb:.1f} KB)")
    else:
        print(f"❌ {filename} が見つかりません")
        all_ok = False

# データの内容確認
if os.path.exists('train_data_natural_language.jsonl'):
    with open('train_data_natural_language.jsonl', 'r', encoding='utf-8') as f:
        data = [json.loads(line) for line in f]
    print(f"\n✓ 学習データ: {len(data)}件 (v1: 35件)")
    
    # サンプルデータ表示
    print("\nサンプルデータ (1件目):")
    print("-" * 70)
    sample = data[0]
    print(f"質問: {sample['input'][:100]}...")
    print(f"回答: {sample['output'][:100]}...")

print("\n" + "=" * 70)
if all_ok:
    print("✅ すべて準備完了! 次のステップに進めます")
else:
    print("⚠️ 不足ファイルがあります。ステップ5に戻ってアップロードしてください")
print("=" * 70)
"""

### 6-3. 実行
▶ボタンをクリック

### 6-4. 結果確認

✓ train_data_natural_language.jsonl (310.5 KB)
✓ config_v3.yaml (2.1 KB)
✓ train_finetuning_v3.py (9.8 KB)
✓ inference_v3.py (7.5 KB)

✓ 学習データ: 150件 (v1: 35件)
"""

## ステップ7: データ確認（v3）

In [ ]:
import json

# 1. データの読み込み
filename = 'train_data_natural_language.jsonl'
with open(filename, 'r', encoding='utf-8') as f:
    data = [json.loads(line) for line in f]

print(f"データ総件数: {len(data)}件")
print("-" * 50)

# 2. サンプルの表示（1件目）
print("✅ 1件目のサンプル確認:")
print(f"【instruction】: {data[0]['instruction']}")
print(f"【input】: {data[0]['input']}")
print(f"【output】:\n{data[0]['output']}")
print("-" * 50)

# 3. フォーマット整合性チェック（日本語の項目が含まれているか）
# チェックしたい項目リスト
required_sections = ["リスクレベル", "該当法", "理由", "修正案"]

format_valid_count = 0
error_samples = []

for i, item in enumerate(data):
    output_text = item.get('output', '')
    
    # すべての必須項目が文字列の中に含まれているか確認
    is_ok = all(section in output_text for section in required_sections)
    
    if is_ok:
        format_valid_count += 1
    else:
        # どの項目が足りないか特定してエラーサンプルとして保持
        missing = [s for s in required_sections if s not in output_text]
        error_samples.append(f"Index {i}: 不足項目 {missing}")

# 4. 結果表示
print(f"レポート形式チェック結果: {format_valid_count}/{len(data)}件")

if format_valid_count == len(data):
    print("すべてのデータが正しい日本語レポート形式で構成されています。学習準備完了！")
else:
    print(f"{len(data) - format_valid_count}件のデータに項目の不足、または表記揺れがあります。")
    if error_samples:
        print("エラーの例 (最初の3件):")
        for err in error_samples[:3]:
            print(f"  {err}")

期待される出力:

データ件数: 150件
サンプル: { "input": "...", "output": "リスクレベル：高\n該当法：...\n理由：...\n修正案：..." }
レポート形式チェック結果: 150/150件 判定: 準備完了（すべてのデータが法務レポート形式に適合しています）

## ステップ8: ファインチューニング実行（v3）

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("=" * 70)
print("Elyza-7B v3 ファインチューニング開始")
print("=" * 70)
print()

!python train_finetuning_v3.py --config_path config_v3.yaml

学習中の表示:

[1/7] モデルとトークナイザーを読み込んでいます...
✓ 完了

[2/7] LoRA設定を適用しています...
学習対象: 8,388,608 パラメータ
LoRAランク: 32

[3/7] データセットを読み込んでいます...
✓ 150件のデータを読み込みました
  - 訓練: 135件
  - 評価: 15件

[4/7] 学習の準備をしています...
✓ 準備完了

--- v3の改善点 ---
1. JSON形式の厳密化
2. エポック数: 7（v2: 5）
3. 学習率: 8e-5（より安定）
4. システムメッセージ強化

[5/7] 学習を開始します...
{'loss': 1.8, 'epoch': 0.1}
{'loss': 1.5, 'epoch': 0.2}
...

⚠️ 学習中はColabを閉じない

## ステップ9: 推論テスト（v3）
学習完了後、新しいセルで実行:

In [ ]:
print("=" * 70)
print("推論テストを実行します")
print("=" * 70)
print()

!python inference_v3.py --mode demo --adapter_path ./results_v3/final_adapter

### 期待される結果:

### [テストケース1] 景品表示法
----------------------------------------------------------------------
入力: 「業界No.1の実績!」という広告を出したいが、特に調査データはない。

結果:
{"リスクレベル":"高","該当法":"景品表示法（優良誤認表示）","理由":"No.1表示には客観的な調査データが必要です。","修正案":"当社調べ（2025年12月時点、主要10社比較）業界最安値クラスなど調査範囲を明記してください。"}

✓ JSON形式: 正常
======================================================================

### 評価サマリー
======================================================================
総テストケース数: 3
JSON形式成功: 3/3 (100.0%)
======================================================================

🔧 うまくいかない場合

ケース1: まだJSON後にテキストが出る
対処法A: さらにエポック数を増やす
config_v3.yamlを編集:

training_arguments:
  num_train_epochs: 10  # 7 → 10

対処法B: データを再確認
すべてのデータがJSON形式のみか確認:

import json

with open('train_data_v3_150samples.jsonl', 'r') as f:
    for i, line in enumerate(f, 1):
        data = json.loads(line)
        output = data['output']
        
        # JSON以外の文字がないかチェック
        try:
            json.loads(output)
        except:
            print(f"行{i}: JSON形式ではありません")
            print(output)

ケース2: JSON形式成功率が50%程度
対処法: 推論パラメータを調整
inference_v3.pyで以下を変更:

response = checker.generate_response(
    instruction, 
    ad_text,
    temperature=0.2,      # 0.3 → 0.2
    top_p=0.85,           # 0.9 → 0.85
)

## ステップ10: 保存とダウンロード

In [ ]:
# ZIP化
!zip -r Tech-Comply-AI-v3.zip results_v3/final_adapter

# ダウンロード
from google.colab import files
files.download('Tech-Comply-AI-v3.zip')